In [1]:

# loading libs
import re
import spacy
import pandas as pd
from wordfreq import zipf_frequency


In [2]:

# nlp = spacy.load("it_core_news_sm") # info below
# # spacy.load(...) takes the name of an installed model package (or a path to one), reads it from disk, and builds the processing pipeline. It returns a Language object, which by convention is stored in a variable called nlp.
# # "it_core_news_md" is the model name, and it breaks down like this:
# # it: the language, Italian
# # core: a general-purpose model (tokenizer, tagger, parser, etc.) rather than a specialized one
# # news: the genre of text it was trained on (newspaper-style writing, plus some other sources)
# # md: the size, medium
# # The sizes are sm (small, about 15 MB), md (medium, about 40 MB), lg (large, about 540 MB) and trf (transformer-based, slowest and most accurate). All sizes share the same pipeline of components. The difference is that md and lg include word vectors, which are useful for similarity comparisons. For sentence splitting, tokenization and lemmatization, sm is often nearly as good.

# # Split sentences reliably on . ! ? (the parser alone sometimes merges them)
# nlp.add_pipe("sentencizer", before="parser")

# # Fixes to spaCy's Italian stopword list:
# # it includes "città" (it shouldn't) and misses some articles
# nlp.vocab["città"].is_stop = False
# for w in ["i", "gli", "le", "lo", "l"]:
#     nlp.Defaults.stop_words.add(w)
#     nlp.vocab[w].is_stop = True

# # Line breaks inside a sentence should not act as separators
# clean_text = re.sub(r"\s+", " ", text).strip()
# doc = nlp(clean_text)

In [3]:
import os
os.environ["PHONEMIZER_ESPEAK_LIBRARY"] = r"C:\Program Files\eSpeak NG\libespeak-ng.dll"
os.environ["PHONEMIZER_ESPEAK_PATH"] = r"C:\Program Files\eSpeak NG\espeak-ng.exe"
from kokoro import KPipeline, KModel
g2p = KPipeline(lang_code="i", model=False)

c:\Users\gmoha\anaconda3\envs\kokoro\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# word level

In [4]:
nlp = spacy.load("it_core_news_sm") 
# # spacy.load(...) takes the name of an installed model package (or a path to one), reads it from disk, and builds the processing pipeline. It returns a Language object, which by convention is stored in a variable called nlp.
nlp.add_pipe("sentencizer", before="parser")
# # Split sentences reliably on . ! ? (the parser alone sometimes merges them)
rows = []
for story in [1,2,3]: 
    for part in [1,2,3]:
        path = f"data/story_{story}_p{part}.txt"
        text = open(path, "r").read()
        # Line breaks inside a sentence should not act as separators
        clean_text = re.sub(r"\s+", " ", text).strip()
        doc = nlp(clean_text)
        for sentence_id, sentence in enumerate(doc.sents, start=1):
            for tok in sentence:
                if tok.is_alpha: # only keep words, not punctuation or numbers
                    rows.append({
                        "story":    story,
                        "part":     part,
                        "sentence": sentence_id,
                        "word":     tok.lower_,
                        "lemma":    tok.lemma_.lower(),
                        "pos":      tok.pos_,        # part of speech: NOUN, VERB, ...
                        "stopword": tok.is_stop,
                        "length":   len(tok.text),   # number of letters
                        "zipf_general": zipf_frequency(tok.text, "it"),
                        "ipa":g2p.g2p(tok.text)[0],
                        "length_ipa":len(g2p.g2p(tok.text)[0])
                    })

words = pd.DataFrame(rows)
words

c:\Users\gmoha\anaconda3\envs\kokoro\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


,story,part,sentence,word,lemma,pos,stopword,length,zipf_general,ipa,length_ipa
0,1,1,1,sara,sara,PROPN,True,4,4.53,sˈaɾa,5
1,1,1,1,lavorava,lavorare,VERB,False,8,4.10,lavorˈava,9
2,1,1,1,come,come,ADP,True,4,6.67,kˈome,5
3,1,1,1,botanica,botanica,NOUN,False,8,3.42,botˈanika,9
4,1,1,1,università,università,NOUN,False,10,5.11,ʊniversitˈa,11
...,...,...,...,...,...,...,...,...,...,...,...
6889,3,3,36,stesso,stesso,ADJ,True,6,5.83,stˈeSo,6
6890,3,3,36,ordine,ordine,NOUN,False,6,5.14,ˈɔrdine,7
6891,3,3,36,usato,usare,VERB,False,5,5.03,ʊzˈato,6
6892,3,3,36,alla,a il,ADP,True,4,6.53,ˈalla,5


## find words with less frequency over all stories

In [5]:
rarest = (
    words
    .drop_duplicates("word")                 # one row per unique word
    .sort_values("zipf_general")             # ascending: rarest first
    .head(200)
)
rarest[["story","part","sentence","word", "lemma", "zipf_general"]]

,story,part,sentence,word,lemma,zipf_general
69,1,1,4,fanura,fanura,0.00
2830,2,1,26,muratrice,muratore,0.00
180,1,1,9,gubbana,gubbana,0.00
4598,3,1,1,valmassa,valmassa,0.00
2327,2,1,2,durna,durna,0.00
...,...,...,...,...,...,...
63,1,1,4,partiva,partire,3.45
2706,2,1,20,costava,costare,3.45
2122,1,3,29,limpido,limpido,3.45
1749,1,3,11,fotografato,fotografare,3.45


# sentence level

In [6]:
sentences = words.groupby(["story", "part", "sentence"], as_index=False).agg(
    n_words=("word", "size"),
    n_tokens=("length_ipa", "sum"),
    avg_word_length=("length", "mean"),
    avg_zipf=("zipf_general", "mean"),
    text=("word", " ".join),
)
sentences = sentences.round(2)
sentences

,story,part,sentence,n_words,n_tokens,avg_word_length,avg_zipf,text
0,1,1,1,19,112,4.89,5.31,sara lavorava come botanica università di lusa...
1,1,1,2,21,125,5.19,5.46,anno doveva raccogliere alcuni campioni di una...
2,1,1,3,21,118,4.81,5.47,isola si chiamava malvara era lunga pochi chil...
3,1,1,4,23,137,5.00,5.48,il traghetto partiva ogni mattina alle sette d...
4,1,1,5,18,114,5.44,5.56,sara aveva preparato il viaggio con attenzione...
...,...,...,...,...,...,...,...,...
330,3,3,32,20,134,5.70,5.33,prima di riconsegnare il piazzale gli operai h...
331,3,3,33,18,103,4.83,5.72,saul è passato in ufficio per portare le osser...
332,3,3,34,16,94,5.19,5.71,il foglio era firmato da quasi tutti i vendito...
333,3,3,35,30,172,4.80,5.68,molti chiedevano di mantenere i passaggi più l...


### too large or small sentence


In [10]:


# short = short.sort_values("n_tokens", ascending=True)
# sort so neighbors are really adjacent, then add previous/next columns
sentences = sentences.sort_values(["story", "part", "sentence"]).reset_index(drop=True)

group = sentences.groupby(["story", "part"])["n_tokens"]
sentences["prev_tokens"] = group.shift(1)    # NaN for the first sentence of a part
sentences["next_tokens"] = group.shift(-1)   # NaN for the last sentence of a part
sentences
def fmt(value):
    return "-" if pd.isna(value) else f"{value:g}"

short = sentences[sentences["n_tokens"] < 85]
print(f"Found {len(short)} sentences with fewer than 85 tokens")
for _, row in short.iterrows():
    print(f"Story {row['story']}, part {row['part']}, sentence {row['sentence']}")
    print(f"  tokens:   {row['n_tokens']:g}  (previous: {fmt(row['prev_tokens'])}, next: {fmt(row['next_tokens'])})")
    print(f"  text:     {row['text']}")
    print()

Found 20 sentences with fewer than 85 tokens
Story 1, part 1, sentence 30
  tokens:   74  (previous: 134, next: 67)
  text:     impiegato ha guardato il mare e ha detto domani il vento dovrebbe calare

Story 1, part 1, sentence 31
  tokens:   67  (previous: 74, next: 121)
  text:     ma oggi il tratto di mare da qui isola è troppo esposto al vento

Story 1, part 2, sentence 16
  tokens:   47  (previous: 114, next: 65)
  text:     alla fine adam ha detto possiamo partire tra ora

Story 1, part 2, sentence 17
  tokens:   65  (previous: 47, next: 104)
  text:     prima devo caricare le casse di frutta per il negozio di malvara

Story 1, part 2, sentence 19
  tokens:   80  (previous: 104, next: 103)
  text:     si sono messi accordo sul prezzo del viaggio e sara ha pagato subito in contanti

Story 1, part 3, sentence 27
  tokens:   58  (previous: 108, next: 41)
  text:     adam ha sorriso e ha detto allora è stato un viaggio utile

Story 1, part 3, sentence 28
  tokens:   41  (previous: 58

# story level

In [14]:
sentences["group"] = sentences["story"].astype(str) + " - " + sentences["part"].astype(str)
stories_part = sentences.groupby("group", as_index=False).agg(
    ave_n_words=("n_words", "mean"),
    n_words=("n_words", "sum"),
    ave_n_tokens=("n_tokens", "mean"),
    ave_word_length=("avg_word_length", "mean"),
    ave_zipf=("avg_zipf", "mean"),
    n_sentences=("sentence", "size"), 
    story=("story", "first"),
    part=("part", "first"),)
stories_part = stories_part.round(2)
stories_part

,group,ave_n_words,n_words,ave_n_tokens,ave_word_length,ave_zipf,n_sentences,story,part
0,1 - 1,20.23,789,113.36,4.68,5.65,39,1,1
1,1 - 2,20.24,749,111.08,4.55,5.67,37,1,2
2,1 - 3,19.95,758,111.08,4.61,5.68,38,1,3
3,2 - 1,21.14,761,117.39,4.62,5.61,36,2,1
4,2 - 2,21.97,769,123.31,4.62,5.66,35,2,2
5,2 - 3,20.13,765,112.89,4.68,5.65,38,2,3
6,3 - 1,20.78,769,123.97,5.05,5.61,37,3,1
7,3 - 2,19.69,768,113.49,4.82,5.68,39,3,2
8,3 - 3,21.28,766,124.00,4.86,5.62,36,3,3


In [16]:
stories = stories_part.groupby("story", as_index=False).agg(
    total_words=("n_words", "sum"),
    total_sentences=("n_sentences", "sum"),
    ave_n_words_per_sentence=("ave_n_words", "mean"),
    
)
stories = stories.round(2)
stories

,story,total_words,total_sentences,ave_n_words_per_sentence
0,1,2296,114,20.14
1,2,2295,109,21.08
2,3,2303,112,20.58
